In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_gold_dimensions
#
# Layer
# -----
# Gold Layer
#
# Purpose
# -------
# Build enterprise conformed dimensions used by all Gold
# fact tables, Semantic Model, Power BI and AI.
#
# Output Dimensions
# -----------------
# • dim_customer
# • dim_country
# • dim_industry
# • dim_rating
# • dim_scenario
# • dim_date
#
# Enterprise Concepts
# -------------------
# ✓ Kimball Dimensional Modelling
# ✓ Conformed Dimensions
# ✓ Star Schema
# ✓ Semantic Model Foundation
# ✓ Enterprise Data Warehouse
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

customer_table = "silver_customer"
rating_table = "silver_rating"
macro_table = "silver_macro"

pipeline_name = "nb_build_gold_dimensions"

run_start_time = datetime.now()

print("ERIP Gold Dimensions Build Started")

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 3, Finished, Available, Finished, False)

ERIP Gold Dimensions Build Started


In [2]:
# ============================================================
# SECTION 2 - READ SILVER BUSINESS ENTITIES
# ============================================================

silver_customer = spark.table(customer_table)
silver_rating = spark.table(rating_table)
silver_macro = spark.table(macro_table)

print(f"Customers : {silver_customer.count()}")
print(f"Ratings   : {silver_rating.count()}")
print(f"Scenarios : {silver_macro.count()}")

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 4, Finished, Available, Finished, False)

Customers : 1000
Ratings   : 1000
Scenarios : 108


In [3]:
# ============================================================
# SECTION 3 - BUILD DIM_CUSTOMER
# ============================================================
#
# Grain
# -----
# One row per Customer
#
# Purpose
# -------
# Enterprise Customer Dimension
#
# Used By
# -------
# • Loan Fact
# • ECL Fact
# • Stress Testing
# • Customer360
# • Executive Dashboards
# ============================================================

dim_customer = (

    silver_customer

    .select(

        "customer_sk",
        "customer_id",
        "customer_name",
        "customer_group_id",
        "parent_company_name",
        "segment",
        "industry_code",
        "industry_name",
        "country_code",
        "country",
        "region",
        "enterprise_revenue_band",
        "customer_risk_category",
        "kyc_risk_rating",
        "esg_score",
        "esg_risk_band",
        "relationship_manager",
        "customer_status",
        "customer_tenure_years"

    )

)

display(dim_customer.limit(10))

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 995d4395-7bb0-4323-a45a-bb35a41be161)

In [4]:
# ============================================================
# SECTION 4 - BUILD DIM_COUNTRY
# ============================================================

dim_country = (

    silver_customer

    .select(

        "country_code",
        "country",
        "region",
        "risk_country"

    )

    .dropDuplicates()

    .withColumn(

        "country_sk",

        row_number().over(

            Window.orderBy("country_code")

        )

    )

)

display(dim_country)

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1f69138-8272-45c0-9c39-0b124f4038ac)

In [5]:
# ============================================================
# SECTION 5 - BUILD DIM_INDUSTRY
# ============================================================

dim_industry = (

    silver_customer

    .select(

        "industry_code",
        "industry_name",
        "nace_code"

    )

    .dropDuplicates()

    .withColumn(

        "industry_sk",

        row_number().over(

            Window.orderBy("industry_code")

        )

    )

)

display(dim_industry)

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c6b27036-ecd6-4234-946a-5227293224fa)

In [6]:
# ============================================================
# SECTION 6 - BUILD DIM_RATING
# ============================================================

dim_rating = (

    silver_rating

    .select(

        "rating_sk",
        "rating_record_id",
        "current_internal_grade",
        "previous_internal_grade",
        "rating_risk_category",
        "pd_band",
        "scorecard_type",
        "model_version",
        "model_override_flag",
        "is_model_override",
        "ifrs9_stage_recommendation",
        "ifrs9_stage_numeric"

    )

)

display(dim_rating.limit(10))

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f5d7990-d1e4-466c-8f4a-8549d48a6993)

In [7]:
# ============================================================
# SECTION 7 - BUILD DIM_SCENARIO
# ============================================================

dim_scenario = (

    silver_macro

    .select(

        "macro_sk",
        "scenario_id",
        "scenario_name",
        "scenario_rank",
        "scenario_severity",
        "scenario_month",
        "economic_cycle",
        "interest_rate_environment",
        "stress_intensity",
        "pd_stress_multiplier",
        "lgd_stress_multiplier"

    )

)

display(dim_scenario)

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb6cbc66-7e41-485c-b772-b22adf33ffc4)

In [8]:
# ============================================================
# SECTION 8 - BUILD DIM_DATE
# ============================================================

dates = (

    silver_macro

    .select(

        col("scenario_month").alias("calendar_date")

    )

    .dropDuplicates()

)

dim_date = (

    dates

    .withColumn("date_sk", date_format("calendar_date","yyyyMMdd").cast("int"))

    .withColumn("year", year("calendar_date"))

    .withColumn("quarter", quarter("calendar_date"))

    .withColumn("month", month("calendar_date"))

    .withColumn("month_name", date_format("calendar_date","MMMM"))

    .withColumn("reporting_period", date_format("calendar_date", "yyyy-MM"))

    .withColumn("reporting_period_sort", year("calendar_date") * 100 + month("calendar_date"))

)

display(dim_date)

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8373593f-79c8-48a5-93d1-150430685943)

In [10]:
# ============================================================
# SECTION 9 - WRITE GOLD DIMENSIONS
# ============================================================

dimensions = {

    "dim_customer": dim_customer,
    "dim_country": dim_country,
    "dim_industry": dim_industry,
    "dim_rating": dim_rating,
    "dim_scenario": dim_scenario,
    "dim_date": dim_date

}

for table_name, df in dimensions.items():

    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(table_name)
    )

    print(f"✓ {table_name} : {df.count()} rows")

print("✓ Gold Dimensions Successfully Created")

StatementMeta(, f43ea36d-778b-40a3-bc53-df15bf20b347, 12, Finished, Available, Finished, False)

✓ dim_customer : 1000 rows
✓ dim_country : 6 rows
✓ dim_industry : 10 rows
✓ dim_rating : 1000 rows
✓ dim_scenario : 108 rows
✓ dim_date : 36 rows
✓ Gold Dimensions Successfully Created
